# Pré-Processamento do PadChest para o Split

O PadChest é um dataset público de radiografias torácicas contendo mais de 160 mil imagens com anotações médicas.
Neste trabalho utilizamos apenas a informação ``ViewPosition_DICOM``, que indica a posição da imagem. Este notebook realiza o pré-processamento do dataset PadChest para um problema de classificação de imagens de pulmão em frente/lado.

As classes consideradas são:

- Frontal (PA / POSTEROANTERIOR)
- Lateral (LATERAL / LL)

Esta tarefa foi escolhida por ser um problema de baixa complexidade semântica, permitindo avaliar se modelos menores conseguem desempenho competitivo.

## Divisão do dataset

O dataset foi dividido em conjuntos de treino e teste usando stratified split.

Isto garante:
- Mesma proporção de classes
- Avaliação justa
- Redução de bias experimental

In [1]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# Caminhos
csv_path = "PADCHEST.csv"
dataset_path = "dataset"
output_path = "data_split"

# Lê o CSV
df = pd.read_csv(csv_path)

# Filtra apenas as imagens de interesse
df = df[df["ViewPosition_DICOM"].isin(["POSTEROANTERIOR", "PA", "LATERAL", "LL"])]

# Normaliza nomes das classes
map_classes = {"POSTEROANTERIOR": "frente", "PA": "frente",
               "LATERAL": "lado", "LL": "lado"}
df["class"] = df["ViewPosition_DICOM"].map(map_classes)

# Agrupa por paciente
patients = df["PatientID"].unique()

# Split pacientes → 70% treino, 15% val, 15% teste
train_patients, test_patients = train_test_split(patients, test_size=0.30, random_state=42)
val_patients, test_patients = train_test_split(test_patients, test_size=0.50, random_state=42)

# Função para salvar arquivos
def copy_images(subset_name, patient_ids):
    subset_df = df[df["PatientID"].isin(patient_ids)]
    for _, row in subset_df.iterrows():
        img_name = row["ImageID"]  
        label = row["class"]
        src = os.path.join(dataset_path, img_name)
        dst = os.path.join(output_path, subset_name, label, img_name)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.exists(src): 
            shutil.move(src, dst)

# Copia arquivos
copy_images("train", train_patients)
copy_images("val", val_patients)
copy_images("test", test_patients)

print("Separação concluída!")

Separação concluída!


## Verificação de integridade das imagens

Datasets médicos, especialmente com tantas imagens, frequentemente contêm arquivos corrompidos.

Implementamos uma verificação usando PIL para detectar:

- arquivos truncados
- erros de leitura
- inconsistências de formato

In [2]:
# import os
# from PIL import Image

# def check_and_remove_corrupted_image(file_path):
#     """
#     Verifica se a imagem está corrompida.
#     Se estiver, remove do disco.
#     """
#     try:
#         with Image.open(file_path) as img:
#             img.verify()  # Verifica integridade do arquivo
#         return False  # Imagem está OK
#     except (IOError, SyntaxError, OSError) as e:  # captura truncadas, corrompidas etc.
#         print(f"Removing corrupted image: {file_path} - {e}")
#         try:
#             os.remove(file_path)
#         except PermissionError:
#             print(f"⚠️ Não foi possível remover (arquivo em uso): {file_path}")
#         return True  # Imagem corrompida removida

# def scan_and_clean_directory(directory):
#     """
#     Percorre todas as subpastas e arquivos do diretório
#     e remove imagens corrompidas.
#     """
#     for root, dirs, files in os.walk(directory):
#         for file in files:
#             file_path = os.path.join(root, file)
#             check_and_remove_corrupted_image(file_path)

# if __name__ == "__main__":
#     # Defina aqui a pasta raiz do seu dataset
#     dataset_dir = "dataset"
    
#     scan_and_clean_directory(dataset_dir)
#     print("✅ Directory scan and cleanup complete.")